# ISIC 2019 External Image Pretraining Launcher

Run this notebook in Google Colab with a GPU runtime. It downloads the official ISIC 2019 training data, maps ISIC labels into the PAD-UFES-20 label space, pretrains the EfficientNet image encoder, then optionally fine-tunes PAD-UFES-20 image-only and multimodal models from that checkpoint.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os
import subprocess

try:
    from google.colab import userdata
except ImportError:
    userdata = None


def get_config(name, default=None):
    value = os.environ.get(name)
    if value:
        return value
    if userdata is not None:
        try:
            return userdata.get(name) or default
        except Exception:
            return default
    return default


def export_config(name, default=None, required=False):
    value = get_config(name, default)
    if required and not value:
        raise RuntimeError(f'Set {name} in Colab Secrets before training.')
    if value:
        os.environ[name] = str(value)
    return value


REPO_URL = 'https://github.com/SalmaneSossey/mlops-teledermatology.git'
BRANCH = 'main'
REPO_DIR = Path('/content/mlops-teledermatology')
HF_DATASET_REPO = get_config('PAD_UFES20_HF_REPO_ID', 'SalmaneExploring/pad-ufes-20')

ISIC_ROOT = Path('/content/isic_2019')
ISIC_IMAGES_ZIP = ISIC_ROOT / 'ISIC_2019_Training_Input.zip'
ISIC_GROUNDTRUTH = ISIC_ROOT / 'ISIC_2019_Training_GroundTruth.csv'
ISIC_IMAGES_DIR = ISIC_ROOT / 'ISIC_2019_Training_Input'
ISIC_SPLITS_DIR = Path('data/processed/isic_2019_splits')
ISIC_PRETRAIN_OUTPUT_DIR = Path('/content/drive/MyDrive/mlops-teledermatology/runs/isic_2019_pretrain')
ISIC_CHECKPOINT = ISIC_PRETRAIN_OUTPUT_DIR / 'efficientnet_b0_best.pt'

PAD_DATA_ROOT = Path('/content/pad_ufes_20')
PAD_IMAGES_DIR = PAD_DATA_ROOT / 'all_images'
PAD_SPLITS_DIR = Path('data/processed/splits')
PAD_IMAGE_ISIC_OUTPUT_DIR = Path('/content/drive/MyDrive/mlops-teledermatology/runs/image_baseline/isic_init')
PAD_MULTIMODAL_ISIC_OUTPUT_DIR = Path('/content/drive/MyDrive/mlops-teledermatology/runs/multimodal_baseline/isic_init')

RUN_ISIC_DOWNLOAD = True
RUN_ISIC_PREPARE = True
RUN_ISIC_PRETRAIN = True
RUN_PAD_DATA_PREPARE = True
RUN_PAD_IMAGE_FINETUNE = True
RUN_PAD_MULTIMODAL_FINETUNE = True

EPOCHS = 8
BATCH_SIZE = 32
IMAGE_SIZE = 224
NUM_WORKERS = 2

export_config('DAGSHUB_TOKEN', required=True)
export_config('DAGSHUB_USERNAME')
export_config('DAGSHUB_REPO_OWNER', 'SalmaneSossey')
export_config('DAGSHUB_REPO_NAME', 'mlops-teledermatology')
export_config('DAGSHUB_MLFLOW_TRACKING_URI')

ISIC_ROOT.mkdir(parents=True, exist_ok=True)
ISIC_PRETRAIN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PAD_IMAGE_ISIC_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PAD_MULTIMODAL_ISIC_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if REPO_DIR.exists():
    subprocess.run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], cwd='/content', check=True)

os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())
print('ISIC root:', ISIC_ROOT)
print('ISIC pretrain output:', ISIC_PRETRAIN_OUTPUT_DIR)
print('PAD data root:', PAD_DATA_ROOT)
print('MLflow tracking URI:', os.environ.get('DAGSHUB_MLFLOW_TRACKING_URI') or f"https://dagshub.com/{os.environ['DAGSHUB_REPO_OWNER']}/{os.environ['DAGSHUB_REPO_NAME']}.mlflow")


In [ ]:
!pip -q install mlflow huggingface_hub scikit-learn


## DagsHub And GPU Checks


In [ ]:
import mlflow
import torch
from mlflow.tracking import MlflowClient

dagshub_token = os.environ.get('DAGSHUB_TOKEN')
dagshub_username = os.environ.get('DAGSHUB_USERNAME')
if dagshub_token and dagshub_username:
    os.environ['MLFLOW_TRACKING_USERNAME'] = dagshub_username
    os.environ['MLFLOW_TRACKING_PASSWORD'] = dagshub_token
elif dagshub_token:
    os.environ['MLFLOW_TRACKING_USERNAME'] = dagshub_token
    os.environ.pop('MLFLOW_TRACKING_PASSWORD', None)

tracking_uri = os.environ.get('DAGSHUB_MLFLOW_TRACKING_URI') or f"https://dagshub.com/{os.environ['DAGSHUB_REPO_OWNER']}/{os.environ['DAGSHUB_REPO_NAME']}.mlflow"
mlflow.set_tracking_uri(tracking_uri)
experiments = MlflowClient(tracking_uri).search_experiments()
print('DagsHub token available:', bool(os.environ.get('DAGSHUB_TOKEN')))
print('MLflow tracking URI:', tracking_uri)
print('Existing experiments:', [(experiment.experiment_id, experiment.name) for experiment in experiments])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
elif RUN_ISIC_PRETRAIN or RUN_PAD_IMAGE_FINETUNE or RUN_PAD_MULTIMODAL_FINETUNE:
    raise RuntimeError('Select a Colab GPU runtime before starting training.')


## Download ISIC 2019


In [ ]:
if RUN_ISIC_DOWNLOAD:
    subprocess.run([
        'wget', '-c',
        'https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_Input.zip',
        '-O', str(ISIC_IMAGES_ZIP),
    ], check=True)
    subprocess.run([
        'wget', '-c',
        'https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_GroundTruth.csv',
        '-O', str(ISIC_GROUNDTRUTH),
    ], check=True)
    subprocess.run(['unzip', '-q', '-n', str(ISIC_IMAGES_ZIP), '-d', str(ISIC_ROOT)], check=True)
else:
    print('Skipping ISIC download.')

print('ISIC metadata exists:', ISIC_GROUNDTRUTH.exists(), ISIC_GROUNDTRUTH)
print('ISIC images dir exists:', ISIC_IMAGES_DIR.exists(), ISIC_IMAGES_DIR)


## Prepare ISIC Splits


In [ ]:
if RUN_ISIC_PREPARE:
    subprocess.run([
        'python', '-m', 'src.data.prepare_isic_2019',
        '--metadata-path', str(ISIC_GROUNDTRUTH),
        '--images-dir', str(ISIC_IMAGES_DIR),
        '--output-dir', str(ISIC_SPLITS_DIR),
        '--image-size', str(IMAGE_SIZE),
    ], check=True)
else:
    print('Skipping ISIC split preparation.')


## Pretrain Image Encoder On ISIC


In [ ]:
if RUN_ISIC_PRETRAIN:
    subprocess.run([
        'python', '-m', 'src.training.train_image_baseline',
        '--images-dir', str(ISIC_IMAGES_DIR),
        '--splits-dir', str(ISIC_SPLITS_DIR),
        '--output-dir', str(ISIC_PRETRAIN_OUTPUT_DIR),
        '--experiment-name', 'pad-ufes-20-isic-2019-pretrain',
        '--hf-dataset-repo', 'ISIC_2019',
        '--epochs', str(EPOCHS),
        '--batch-size', str(BATCH_SIZE),
        '--image-size', str(IMAGE_SIZE),
        '--num-workers', str(NUM_WORKERS),
        '--sampler', 'weighted_random',
    ], check=True)
else:
    print('Skipping ISIC pretraining.')

print('Expected ISIC checkpoint:', ISIC_CHECKPOINT)
print('Checkpoint exists:', ISIC_CHECKPOINT.exists())


## Prepare PAD-UFES-20 Data


In [ ]:
if RUN_PAD_DATA_PREPARE:
    subprocess.run([
        'python', '-m', 'src.data.download_pad_ufes_20',
        '--repo-id', HF_DATASET_REPO,
        '--output-dir', str(PAD_DATA_ROOT),
        '--force',
    ], check=True)
    subprocess.run([
        'python', '-m', 'src.data.make_image_splits',
        '--metadata-path', str(PAD_DATA_ROOT / 'metadata.csv'),
        '--images-dir', str(PAD_IMAGES_DIR),
        '--output-dir', str(PAD_SPLITS_DIR),
    ], check=True)
else:
    print('Skipping PAD data preparation.')


## Fine-Tune PAD Image Model From ISIC Checkpoint


In [ ]:
if RUN_PAD_IMAGE_FINETUNE:
    subprocess.run([
        'python', '-m', 'src.training.train_image_baseline',
        '--images-dir', str(PAD_IMAGES_DIR),
        '--splits-dir', str(PAD_SPLITS_DIR),
        '--output-dir', str(PAD_IMAGE_ISIC_OUTPUT_DIR),
        '--experiment-name', 'pad-ufes-20-image-baseline-isic-init',
        '--hf-dataset-repo', HF_DATASET_REPO,
        '--initial-checkpoint', str(ISIC_CHECKPOINT),
        '--epochs', str(EPOCHS),
        '--batch-size', str(BATCH_SIZE),
        '--image-size', str(IMAGE_SIZE),
        '--num-workers', str(NUM_WORKERS),
    ], check=True)
else:
    print('Skipping PAD image fine-tune.')


## Fine-Tune PAD Multimodal Model From ISIC Checkpoint


In [ ]:
if RUN_PAD_MULTIMODAL_FINETUNE:
    subprocess.run([
        'python', '-m', 'src.training.train_multimodal_baseline',
        '--images-dir', str(PAD_IMAGES_DIR),
        '--metadata-path', str(PAD_DATA_ROOT / 'metadata.csv'),
        '--splits-dir', str(PAD_SPLITS_DIR),
        '--output-dir', str(PAD_MULTIMODAL_ISIC_OUTPUT_DIR),
        '--experiment-name', 'pad-ufes-20-image-baseline-multimodal-isic-init',
        '--hf-dataset-repo', HF_DATASET_REPO,
        '--initial-image-checkpoint', str(ISIC_CHECKPOINT),
        '--epochs', str(EPOCHS),
        '--batch-size', str(BATCH_SIZE),
        '--image-size', str(IMAGE_SIZE),
        '--num-workers', str(NUM_WORKERS),
    ], check=True)
else:
    print('Skipping PAD multimodal fine-tune.')


## What To Compare

After the runs finish, compare the DagsHub MLflow test metrics against the existing PAD-only image and multimodal baselines. Keep the ISIC-initialized model only if it improves PAD-UFES-20 test macro F1, balanced accuracy, high-risk recall, or SCC/MEL behavior without damaging another high-risk class too much.
